# 04 · Part C: for which patients does a GLP-1 / SGLT2 drug pay for itself?

**Why this question matters in the US:** the doctor chooses the drug, but the **payer** (the patient's insurance plan) decides who can actually get it: whether it's covered, whether the doctor must file a prior authorization, whether the patient must fail cheaper drugs first (step therapy), and what the patient pays at the pharmacy. Payers loosen those hurdles only where they expect the drug to pay for itself. So the useful question is: **for which patients do the hospital stays the drug prevents cover what it costs the payer?**

| Input | From | Value |
|---|---|---|
| How often each patient was in hospital **before** this stay | Part A, notebook 02 (prior stays in the past year) | per patient |
| What a year of the drug costs | Part B, notebook 03 (Jardiance 2024 gross vs. 2026 negotiated; semaglutide negotiated) | 3 prices |
| How much the drug cuts hospital stays | EMPA-REG OUTCOME trial: all-cause hospitalization HR 0.89 → **11%** fewer (95% CI 4–18%) | `reference/assumptions.csv` |
| What a hospital stay costs Medicare | CMS inpatient payments (Medicare's share, 2024), in **2026 dollars** | section 0 |

**The model, per patient per year**

```
admissions avoided  = expected admissions next year × 11%
expected admissions = prior hospital stays in the past year   (the current stay is NOT counted:
                                                                a drug started at discharge can't prevent it)
savings             = admissions avoided × Medicare payment per stay (2026 $)
net value           = savings − annual drug cost            (positive = pays for itself)
break-even          = annual drug cost ÷ (11% × payment per stay)    prior stays a year
```

**Assumptions stated up front**
- Next year's admissions match last year's prior stays. Frequent users drift toward average (regression to the mean), so I test half as many. Section 3 checks this against what actually happened after discharge.
- The 11% comes from EMPA-REG, a trial in outpatients with type 2 diabetes and heart disease, measuring time to the **first** hospitalization. Applying it to admission counts in all diabetic inpatients is an extrapolation, so I test 4% and the heart-disease subgroup.
- Only hospital savings count. Fewer deaths and slower kidney decline aren't priced, so value is understated.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # make src/ importable

import json
import numpy as np
import pandas as pd
from src import apis
from src.config import ROOT, PROCESSED, RESULTS, update_results

p = pd.read_parquet(PROCESSED / "patients_scored.parquet")
A = pd.read_csv(ROOT / "reference" / "assumptions.csv").set_index("parameter")
R_ALL = json.loads(RESULTS.read_text())
ANNUAL = R_ALL["lever"]["annual_cost"]                     # from Part B (notebook 03)
RRR = A.loc["rrr_hospitalization", ["low", "base", "high"]].astype(float)
SG_N, SG_G, GL_N = "SGLT2 · negotiated (2026)", "SGLT2 · gross 2024", "GLP-1 · negotiated (2027)"
pd.Series(ANNUAL).round(0).rename("annual drug cost ($)")

## 0. What a hospital stay costs Medicare, in 2026 dollars

The inputs come from different years (hospital patients 1999–2008, drug data 2013–2024, trial 2015, prices 2026/27). The decision is about 2026, so every dollar is put in 2026 terms.

- **Base:** Medicare's **own share** of the average payment per stay across all 2024 admissions (CMS `Avg_Mdcr_Pymt_Amt`), moved to 2026 with the BLS price index for inpatient hospital services. The CPI is a proxy: Medicare's own rate updates are usually smaller, so the low case uses no inflation at all.
- **Low:** Medicare's share for diabetes and heart-failure stays only, in 2024 dollars.
- **High:** the **total** payment per stay (Medicare plus patient cost-sharing), relevant for a plan that carries the whole cost. AHRQ's 2018 all-payer readmission cost is the cross-check.

In [ ]:
inp = apis.cms_inpatient_national()
cpi = apis.bls_series(apis.CPI_INPATIENT, 2017, 2026)
level = cpi.groupby("year").value.mean()                   # 2026 = January-October average
to_2026 = lambda year: level[2026] / level[year]
w = lambda d, col: (d.Tot_Dschrgs * d[col]).sum() / d.Tot_Dschrgs.sum()
DIAB_HF = ["637", "638", "639", "291", "292", "293"]        # diabetes and heart-failure DRGs

COST = pd.Series({
    "low": w(inp[inp.DRG_Cd.isin(DIAB_HF)], "Avg_Mdcr_Pymt_Amt"),          # 2024 $, no inflation
    "base": w(inp, "Avg_Mdcr_Pymt_Amt") * to_2026(2024),
    "high": w(inp, "Avg_Tot_Pymt_Amt") * to_2026(2024),
})
hcup_2026 = A.loc["cost_per_admission_hcup_2018", "base"] * to_2026(2018)
recon = pd.DataFrame([
    {"measure": "Medicare's share per stay, all stays (base)", "2024_$": w(inp, "Avg_Mdcr_Pymt_Amt"), "2026_$": COST["base"]},
    {"measure": "Medicare's share, diabetes + heart-failure stays (low, not inflated)", "2024_$": COST["low"], "2026_$": COST["low"]},
    {"measure": "Total payment per stay, all stays (high)", "2024_$": w(inp, "Avg_Tot_Pymt_Amt"), "2026_$": COST["high"]},
    {"measure": "AHRQ HCUP 2018 all-payer readmission cost (cross-check)", "2024_$": np.nan, "2026_$": hcup_2026},
]).round(0)
print(f"Hospital price index: +{to_2026(2018)-1:.0%} from 2018, +{to_2026(2024)-1:.0%} from 2024")
recon

In [ ]:
C1 = R_ALL["cleaning"]
bench = A.loc["benchmark_readmission_diabetes", ["low", "high"]].astype(float)
TIME = pd.DataFrame([
    {"input": "Prior hospital stays per patient", "data_year": "1999–2008",
     "how_aligned": "Checked against what happened after discharge (section 3) and recent readmission rates; 50% scenario",
     "effect_on_result": f"Likely understates value: 30-day readmission here ({100*C1['readmit_rate_all_stays']:.1f}% of stays) is below recent studies ({100*bench['low']:.0f}–{100*bench['high']:.0f}%)"},
    {"input": "Drug effect (11%)", "data_year": "2015 trial", "how_aligned": "Range 4–18%; heart-disease subgroup tested",
     "effect_on_result": "Could overstate: the trial measured first hospitalization in outpatients with heart disease"},
    {"input": "Cost of a stay", "data_year": "2024 (CMS); 2018 (HCUP check)", "how_aligned": "Medicare's share, moved to 2026 with the BLS hospital price index",
     "effect_on_result": "CPI may run above Medicare's own rate updates; low case uses no inflation"},
    {"input": "Drug prices", "data_year": "2024 gross; 2026/27 negotiated", "how_aligned": "Negotiated prices are already the decision years",
     "effect_on_result": "Gross is before confidential rebates, so it overstates the net price"},
])
TIME

## 1. The break-even point

How many **prior** hospital stays a year a patient needs for the drug to pay for itself, at each price.

In [ ]:
def breakeven(annual, rrr=RRR["base"], cost=COST["base"]):
    return annual / (rrr * cost)
be = pd.DataFrame({"annual drug cost ($)": pd.Series(ANNUAL), "break-even prior stays / year": pd.Series({k: breakeven(v) for k, v in ANNUAL.items()})}).round(2)
be

## 2. Who clears it: patients grouped by prior hospital stays in the past year

In [ ]:
def net_value(prior, annual, rrr=RRR["base"], cost=COST["base"], factor=1.0):
    return prior * factor * rrr * cost - annual

p["stays_group"] = pd.cut(p.prior_stays, [-1, 0, 1, 2, 100], labels=["No prior stay", "1 prior stay", "2 prior stays", "3+ prior stays"])
grp = (p.groupby("stays_group", observed=True)
         .agg(patients=("prior_stays", "size"), prior_stays=("prior_stays", "mean"),
              readmit_30_pct=("readmit_30", "mean"), later_readmit_pct=("any_readmit", "mean"), heart_pct=("has_heart_dx", "mean")))
grp["share_pct"] = 100 * grp.patients / len(p)
grp[["readmit_30_pct", "later_readmit_pct", "heart_pct"]] *= 100
for k, v in ANNUAL.items():
    grp[k] = net_value(grp.prior_stays, v)
grp.round(1)

**Net value per patient-year** (positive = pays for itself). At the negotiated SGLT2 price the break-even is about **1.26 prior stays a year**:
- **No prior stay (88% of patients):** the drug can't pay for itself through hospital savings. Treating them is a clinical decision.
- **1 prior stay:** slightly below break-even, so roughly cost-neutral.
- **2+ prior stays (3.4%):** clearly pays for itself, and the margin grows with each stay.

At 2024 gross prices only patients with 3+ prior stays pay off.

## 3. Does the past predict the future? Checking against what happened after discharge

The data records whether each patient came back later (`readmitted`: within 30 days, after 30 days, or not at all). It's a yes/no, not a count, so it can't replace the admission count, but it tests the key assumption: **do patients with more prior stays actually return more often?**

In [ ]:
check = grp[["patients", "prior_stays", "readmit_30_pct", "later_readmit_pct"]].round(1)
check

Yes, steadily: about 38% of patients with no prior stay came back at some point, against roughly two-thirds of patients with 2+ prior stays, and their 30-day readmission rate is two to three times higher. Prior stays are a valid forward signal. The size of the future count is still an assumption, which is why section 5 tests half as many.

## 4. The rule: two or more prior stays in the past year

Sized per 1,000 diabetic discharges, at each price.

In [ ]:
def rule_summary(q, annual, factor=1.0, rrr=RRR["base"], cost=COST["base"]):
    n = 1000 * len(q) / len(p)
    avoided = n * q.prior_stays.mean() * factor * rrr
    return {"qualifying per 1,000 discharges": n, "admissions avoided / yr": avoided, "drug cost / yr ($)": n * annual,
            "hospital savings / yr ($)": avoided * cost, "net / yr ($)": avoided * cost - n * annual}

q2 = p[p.prior_stays >= 2]
q1 = p[p.prior_stays == 1]
rule = pd.DataFrame({k: rule_summary(q2, v) for k, v in ANNUAL.items()}).T
print(f"Qualifying (2+ prior stays): {len(q2):,} of {len(p):,} ({len(q2)/len(p):.1%}); "
      f"{q2.readmit_30.sum()/p.readmit_30.sum():.0%} of all 30-day readmissions; heart diagnosis {q2.has_heart_dx.mean():.0%}")
rule.round(0)

## 4b. Making the rule precise: tiers, clinical criteria, and the price that makes it work

A payer writes coverage criteria as **clinical indication + utilization + price**. Three refinements, each tested:

1. **Price tiers.** For each group, the **monthly net price** at which coverage pays for itself = prior stays × 11% × cost of a stay ÷ 12. This turns the rule into a contract term.
2. **Clinical indication.** Guidelines (and the EMPA-REG trial) target diabetic patients with heart or kidney disease. Does adding that criterion change who comes back?
3. **Adding the model.** For patients with only one prior stay, does a high model risk score identify ones who behave like the 2+ group?

In [ ]:
tiers = (p.groupby("stays_group", observed=True)
           .agg(patients=("prior_stays", "size"), prior_stays=("prior_stays", "mean"), later_readmit_pct=("any_readmit", "mean"))
           .assign(share_pct=lambda d: 100 * d.patients / len(p), later_readmit_pct=lambda d: 100 * d.later_readmit_pct,
                   breakeven_price_month=lambda d: d.prior_stays * RRR["base"] * COST["base"] / 12))
NEG_MONTH = ANNUAL[SG_N] / 12
tiers["at the negotiated SGLT2 price"] = np.where(tiers.breakeven_price_month >= NEG_MONTH, "pays for itself", "costs more than it saves")
tiers.round(1)

In [ ]:
hi = p.risk_score >= p.risk_score.quantile(0.8)
card = (p.has_heart_dx == 1) | (p.has_kidney_dx == 1)
masks = {
    "2+ prior stays": p.prior_stays >= 2,
    "2+ prior stays, heart or kidney diagnosis": (p.prior_stays >= 2) & card,
    "2+ prior stays, no heart or kidney diagnosis": (p.prior_stays >= 2) & ~card,
    "1 prior stay": p.prior_stays == 1,
    "1 prior stay, model's top 20% risk": (p.prior_stays == 1) & hi,
    "1 prior stay, rest": (p.prior_stays == 1) & ~hi,
}
refine = pd.DataFrame([{"group": g, "patients": int(m.sum()), "share_pct": 100 * m.mean(),
                        "later_readmit_pct": 100 * p.loc[m, "any_readmit"].mean(), "readmit_30_pct": 100 * p.loc[m, "readmit_30"].mean()}
                       for g, m in masks.items()]).round(1)
refine

**What the refinements show**
- **Price is the strongest lever.** Each tier has a monthly net price below which covering the drug pays for itself (table above). Patients with 2 prior stays clear the $197 negotiated price; patients with 3+ clear it by a wide margin, close to the 2024 gross price; patients with 1 prior stay don't. The manufacturer can offer a net price, or an outcomes-based rebate, per tier.
- **Heart or kidney disease doesn't change how often patients come back** (similar later-readmission rates with and without it), but it's the guideline indication and the trial's population, so it belongs in the coverage criteria for clinical reasons, not economic ones.
- **The model adds little on top of the rule.** Among one-prior-stay patients, a high risk score raises later readmission only modestly, well short of the 2+ group. So I don't add it.

**The refined rule a plan could write:** *type 2 diabetes with heart or kidney disease, 2 or more hospital stays in the past 12 months, and not already on a GLP-1 or SGLT2 drug → cover with no prior authorization or step therapy at the negotiated price. With 3+ stays, the case holds even near gross price.* Clinical exclusions (for example type 1 diabetes, or low kidney function for some agents) stay with the prescriber.

## 5. How sure are we? Stress tests

Net value per patient-year for the target group (2+ prior stays) and, for comparison, everyone with 1+ prior stay, at the negotiated SGLT2 price.

In [ ]:
base_cost = ANNUAL[SG_N]
def stress(q):
    m = q.prior_stays.mean()
    return {"base": net_value(m, base_cost),
            "effect 4% (trial low end)": net_value(m, base_cost, rrr=RRR["low"]),
            "future stays = 50% of past": net_value(m, base_cost, factor=0.5),
            "cost of a stay: diabetes/heart-failure, 2024 $": net_value(m, base_cost, cost=COST["low"]),
            "2024 gross price": net_value(m, ANNUAL[SG_G]),
            "combined downside (50% future + low cost)": net_value(m, base_cost, factor=0.5, cost=COST["low"])}
STRESS = pd.DataFrame({
    "2+ prior stays": stress(q2),
    "2+ prior stays, Medicare-insured only": stress(q2[q2.payer_code == "MC"]),
    "2+ prior stays, heart diagnosis only": stress(q2[q2.has_heart_dx == 1]),
    "1+ prior stays": stress(p[p.prior_stays >= 1]),
}).T.round(0)
STRESS

In [ ]:
m2 = q2.prior_stays.mean()
tornado = pd.DataFrame([
    {"assumption": "Hospitalization reduction (4% – 18%)", "low": net_value(m2, base_cost, rrr=RRR["low"]), "high": net_value(m2, base_cost, rrr=RRR["high"])},
    {"assumption": "Future stays (50% – 100% of past)", "low": net_value(m2, base_cost, factor=0.5), "high": net_value(m2, base_cost)},
    {"assumption": f"Cost of a stay (${COST['low']:,.0f} – ${COST['high']:,.0f})", "low": net_value(m2, base_cost, cost=COST["low"]), "high": net_value(m2, base_cost, cost=COST["high"])},
    {"assumption": "Price (negotiated – 2024 gross)", "low": net_value(m2, ANNUAL[SG_G]), "high": net_value(m2, base_cost)},
]).assign(swing=lambda d: (d.high - d.low).abs()).sort_values("swing", ascending=False).round(0)
tornado

**Reading the stress tests.** For patients with 2+ prior stays, the drug stays positive if future stays are half the past or if a stay costs only what diabetes and heart-failure stays cost. It turns negative at the **low end of the trial's effect range (4%)**, at the **2024 gross price**, and when two pessimistic assumptions combine. Results are nearly identical for Medicare-insured patients and for patients with a heart diagnosis (the trial's population). For the broader 1+ group, almost every pessimistic assumption turns it negative, which is why the rule is 2+, not 1+.

## 6. Rule or model? Compared on what actually happened next

Part A built a readmission model. Here it competes with the rule. Both pick the **same number of patients** (the 2+ group's size). They're compared on **outcomes after discharge** only, not on prior stays, which would favor the rule by construction.

In [ ]:
k = len(q2)
by_model = p.nlargest(k, "risk_score")
cmp = pd.DataFrame([
    {"selection": "Rule: 2+ prior stays", "patients": k, "later_readmit_pct": 100 * q2.any_readmit.mean(),
     "readmit_30_pct": 100 * q2.readmit_30.mean(), "share_of_30d_readmissions_pct": 100 * q2.readmit_30.sum() / p.readmit_30.sum()},
    {"selection": f"Model: top {k/len(p):.1%} predicted risk", "patients": k, "later_readmit_pct": 100 * by_model.any_readmit.mean(),
     "readmit_30_pct": 100 * by_model.readmit_30.mean(), "share_of_30d_readmissions_pct": 100 * by_model.readmit_30.sum() / p.readmit_30.sum()},
]).round(1)
overlap = len(set(by_model.index) & set(q2.index)) / k
print(f"Patients picked by both: {overlap:.0%}")
cmp

They pick mostly **different** patients. The model, trained for 30-day readmission, finds more of those (25% vs. 22%). The rule finds patients who **keep** coming back (71% vs. 58% readmitted at some point after discharge), and the drug's value comes from stays prevented over a whole year. So the rule picks the patients the drug actually pays off in, and a payer can apply it in any claims system without a model.

## 7. GLP-1 drugs: what effect would they need?

The 11% comes from an SGLT2 trial. GLP-1 trials report fewer heart attacks and heart-failure admissions, but not a comparable all-cause hospitalization estimate, so instead of assuming one I ask what effect a GLP-1 at its negotiated price would need to pay for itself.

In [ ]:
needed = pd.Series({
    "2+ prior stays": ANNUAL[GL_N] / (q2.prior_stays.mean() * COST["base"]),
    "1 prior stay": ANNUAL[GL_N] / (1.0 * COST["base"]),
}).rename("all-cause hospitalization reduction needed")
(100 * needed).round(1).astype(str) + "%"

## 8. Who captures the savings?

In **traditional Medicare**, the drug is paid by **Part D** (a drug plan) while the avoided hospital stays are paid by **Part A**, so no single party sees both sides. The organizations that pay for **both** the drug and the hospital stay, and so capture the net value, are **Medicare Advantage plans with drug coverage** and **risk-bearing groups such as ACOs**. They're the audience for this value case, and they control coverage hurdles (prior authorization, step therapy, copay tier).

In [ ]:
update_results("value", {
    "cost": {k: round(v) for k, v in COST.items()}, "recon": recon.fillna("").to_dict("records"), "time_table": TIME.to_dict("records"),
    "cpi_2018_2026": round(to_2026(2018) - 1, 3), "cpi_2024_2026": round(to_2026(2024) - 1, 3),
    "annual_cost": ANNUAL, "breakeven": be["break-even prior stays / year"].to_dict(),
    "groups": grp.reset_index().astype({"stays_group": str}).round(1).to_dict("records"),
    "rule": rule.reset_index().rename(columns={"index": "scenario"}).round(0).to_dict("records"),
    "target": {"n": int(len(q2)), "pct": round(100 * len(q2) / len(p), 1), "readmit_30": round(100 * q2.readmit_30.mean(), 1),
               "readmit_30_rest": round(100 * p[p.prior_stays < 2].readmit_30.mean(), 1), "later_readmit": round(100 * q2.any_readmit.mean(), 1),
               "share_of_readmissions": round(100 * q2.readmit_30.sum() / p.readmit_30.sum(), 1), "heart_pct": round(100 * q2.has_heart_dx.mean(), 1),
               "mean_prior": round(q2.prior_stays.mean(), 2), "one_prior_net": round(float(net_value(1.0, base_cost))),
               "medicare_n": int((q2.payer_code == "MC").sum())},
    "stress": {k: {kk: float(vv) for kk, vv in v.items()} for k, v in STRESS.T.to_dict().items()},
    "tornado": tornado.to_dict("records"), "base_net": float(STRESS.loc["2+ prior stays", "base"]),
    "rule_vs_model": cmp.to_dict("records"), "rule_model_overlap": round(100 * overlap, 1),
    "glp1_needed": {k: round(100 * x, 1) for k, x in needed.items()},
    "tiers": tiers.reset_index().astype({"stays_group": str}).round(1).to_dict("records"),
    "refine": refine.to_dict("records"),
})
print("saved")